# 12 — Calibración de los clasificadores (M1 y M2)

**Objetivo.** Evaluar si las probabilidades que los modelos entregan a la aplicación son fiables como medida de
confianza, dado que la interfaz muestra ese valor al usuario junto con el diagnóstico.

**Tecnología.** TensorFlow/Keras para la inferencia de doble entrada, NumPy y Matplotlib para el análisis; se emplean
los modelos ya entrenados y el conjunto de prueba independiente, sin reentrenamiento.

**Metodología.** Sobre el conjunto de prueba se obtienen las probabilidades predichas con la misma doble entrada del
entrenamiento (imagen original y hoja aislada por M_seg, con normalización Shades-of-Gray). La calidad de la
calibración se cuantifica con el error de calibración esperado (ECE) y máximo (MCE), calculados por agrupamiento de
la confianza en intervalos equiespaciados, y con el puntaje de Brier. Se acompaña de diagramas de confiabilidad que
contrastan la confianza declarada con la exactitud observada.

In [ ]:
!pip install -q tensorflow albumentations scikit-learn pandas matplotlib

In [ ]:
from pathlib import Path
import glob, shutil, zipfile
from google.colab import drive

drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/glycine_vision_baselines')
OUT.mkdir(parents=True, exist_ok=True)
DRIVE = Path('/content/drive/MyDrive')


def _first(candidatos, patron=None):
    for c in candidatos:
        if Path(c).exists():
            return Path(c)
    if patron:
        hits = sorted(glob.glob(patron, recursive=True))
        if hits:
            return Path(hits[0])
    return None


SPLIT = _first(['/content/splits'])
if SPLIT is None:
    _zip = _first([OUT / 'splits.zip'], str(DRIVE / '**' / 'splits.zip'))
    if _zip is not None:
        shutil.copy(_zip, '/content/splits.zip')
        with zipfile.ZipFile('/content/splits.zip') as z:
            z.extractall('/content')
        SPLIT = Path('/content/splits')
    else:
        SPLIT = _first([], str(DRIVE / '**' / 'splits'))
assert SPLIT is not None, 'No se encontró la carpeta splits ni splits.zip en Drive.'

M1_PATH = _first([OUT / 'model1_binary.keras', '/content/outputs/model1_binary.keras'], str(DRIVE / '**' / 'model1_binary.keras'))
M2_PATH = _first([OUT / 'model2_pathogen.keras', '/content/outputs/model2_pathogen.keras'], str(DRIVE / '**' / 'model2_pathogen.keras'))
SEG_PATH = _first([OUT / 'model_seg.keras', '/content/outputs/model_seg.keras'], str(DRIVE / '**' / 'model_seg.keras'))
TEST_BIN = _first([SPLIT / 'test' / 'clasificacion_binaria'], str(SPLIT / '**' / 'clasificacion_binaria'))
TEST_PAT = _first([SPLIT / 'test' / 'clasificacion_patogeno'], str(SPLIT / '**' / 'clasificacion_patogeno'))

for _n, _p in [('model1_binary.keras', M1_PATH), ('model2_pathogen.keras', M2_PATH),
               ('model_seg.keras', SEG_PATH), ('test/clasificacion_binaria', TEST_BIN),
               ('test/clasificacion_patogeno', TEST_PAT)]:
    assert _p is not None, f'No encontrado: {_n}'
print('Splits:', SPLIT)
print('M1:', M1_PATH)
print('M2:', M2_PATH)
print('M_seg:', SEG_PATH)
print('Test binario:', TEST_BIN)
print('Test patógeno:', TEST_PAT)

In [ ]:
import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.applications.efficientnet import preprocess_input

m1 = tf.keras.models.load_model(M1_PATH, compile=False)
m2 = tf.keras.models.load_model(M2_PATH, compile=False)
_MSEG = tf.keras.models.load_model(SEG_PATH, compile=False)


def chromatic_normalize(img_rgb):
    x = img_rgb.astype(np.float32)
    il = np.power(np.mean(np.power(x, 6), axis=(0, 1)), 1.0 / 6.0)
    return np.clip(x * np.clip(il.mean() / (il + 1e-6), 0.6, 1.6), 0, 255).astype(np.uint8)


def mseg_mask(img_rgb, size):
    small = chromatic_normalize(cv2.resize(img_rgb, (256, 256)))
    prob = _MSEG.predict((small.astype(np.float32) / 255.0)[np.newaxis], verbose=0)[0]
    leaf = (np.argmax(prob, -1) == 1).astype(np.uint8)
    return cv2.resize(leaf, size, interpolation=cv2.INTER_NEAREST)


def _list(directory):
    classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
    idx = {c: i for i, c in enumerate(classes)}
    items = []
    for c in classes:
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
            for fp in (Path(directory) / c).glob(ext):
                items.append((str(fp), idx[c]))
    return idx, items


def predict_dual(model, directory, size, batch=32):
    idx, items = _list(directory)
    probs, ytrue = [], []
    ob, lb, yb = [], [], []

    def flush():
        if not ob:
            return
        probs.append(model.predict([np.stack(ob), np.stack(lb)], verbose=0))
        ytrue.extend(yb)

    for fp, y in items:
        img = np.array(Image.open(fp).convert('RGB').resize(size))
        norm = chromatic_normalize(img)
        iso = norm.copy(); iso[mseg_mask(img, size) == 0] = 0
        ob.append(preprocess_input(norm.astype(np.float32)))
        lb.append(preprocess_input(iso.astype(np.float32)))
        yb.append(y)
        if len(ob) == batch:
            flush(); ob, lb, yb = [], [], []
    flush()
    return idx, np.array(ytrue), np.concatenate(probs) if probs else np.zeros((0, 1))


idx1, y1, p1 = predict_dual(m1, TEST_BIN, (240, 240))
idx2, y2, p2 = predict_dual(m2, TEST_PAT, (224, 224))
print('M1:', p1.shape, '| M2:', p2.shape)

In [ ]:
def calibration_stats(conf, correct, n_bins=10):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece, mce, rows = 0.0, 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if not m.any():
            rows.append((lo, hi, 0, np.nan, np.nan))
            continue
        acc = float(correct[m].mean())
        avg = float(conf[m].mean())
        gap = abs(acc - avg)
        ece += m.mean() * gap
        mce = max(mce, gap)
        rows.append((lo, hi, int(m.sum()), acc, avg))
    return float(ece), float(mce), rows


def reliability_plot(rows, titulo, archivo):
    centers = [(lo + hi) / 2 for lo, hi, n, a, c in rows]
    accs = [a for lo, hi, n, a, c in rows]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='calibración perfecta')
    ax.plot(centers, accs, 'o-', color='steelblue', label='observado')
    ax.set_xlabel('Confianza media'); ax.set_ylabel('Exactitud observada')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_title(titulo)
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT / archivo, dpi=120); plt.show()


enf_idx = next((i for c, i in idx1.items() if 'enferm' in c.lower()), 0)
p_pos = p1.reshape(-1)
p_enf = p_pos if enf_idx == 1 else 1.0 - p_pos
y_enf = (y1 == enf_idx).astype(int)
conf1 = np.maximum(p_enf, 1.0 - p_enf)
correct1 = ((p_enf >= 0.5).astype(int) == y_enf)
brier1 = float(np.mean((p_enf - y_enf) ** 2))
ece1, mce1, rows1 = calibration_stats(conf1, correct1)

conf2 = p2.max(axis=1)
correct2 = (p2.argmax(axis=1) == y2)
onehot = np.eye(p2.shape[1])[y2]
brier2 = float(np.mean(np.sum((p2 - onehot) ** 2, axis=1)))
ece2, mce2, rows2 = calibration_stats(conf2, correct2)

import pandas as pd
tabla = pd.DataFrame([
    {'Modelo': 'M1 (estado sanitario)', 'ECE': round(ece1, 4), 'MCE': round(mce1, 4),
     'Brier': round(brier1, 4), 'Confianza media': round(float(conf1.mean()), 4),
     'Exactitud': round(float(correct1.mean()), 4)},
    {'Modelo': 'M2 (patógeno)', 'ECE': round(ece2, 4), 'MCE': round(mce2, 4),
     'Brier': round(brier2, 4), 'Confianza media': round(float(conf2.mean()), 4),
     'Exactitud': round(float(correct2.mean()), 4)},
])
tabla.to_csv(OUT / 'calibracion.csv', index=False)
tabla

In [ ]:
reliability_plot(rows1, 'M1 (estado sanitario)', 'calibracion_m1.png')
reliability_plot(rows2, 'M2 (patógeno)', 'calibracion_m2.png')

In [ ]:
import json
for nombre, ece, conf, acc in [('M1', ece1, float(conf1.mean()), float(correct1.mean())),
                               ('M2', ece2, float(conf2.mean()), float(correct2.mean()))]:
    if ece < 0.05:
        estado = 'bien calibrado'
    elif ece < 0.10:
        estado = 'calibración aceptable'
    else:
        estado = 'requiere recalibración (p. ej. temperature scaling)'
    sesgo = 'sobreconfiado' if conf > acc else 'subconfiado'
    print(f'{nombre}: ECE = {ece:.4f} -> {estado}; el modelo es {sesgo} (confianza {conf:.3f} vs exactitud {acc:.3f}).')

json.dump({'m1': {'ece': ece1, 'mce': mce1, 'brier': brier1},
           'm2': {'ece': ece2, 'mce': mce2, 'brier': brier2}},
          open(OUT / 'calibracion.json', 'w'), indent=2, ensure_ascii=False)